# Credit risk prediction — logistic regression

The pipeline (features, encoding, model) lives in `model.py` and is shared with `train_model.py` and the bot.

In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

from model import RAW_FEATURES, TARGET, build_pipeline
from train_model import best_threshold, evaluate

ModuleNotFoundError: No module named 'pandas'

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(f"Training data: {train_df.shape}, Test data: {test_df.shape}")
print(f"Total missing values: {train_df.isnull().sum().sum() + test_df.isnull().sum().sum()}")

Training data: (58645, 13), Test data: (39098, 12)
Total missing values: 0


## Target

`loan_status = 1` means **high risk (default)**: its rate grows from grade A to G and is higher for clients with a previous default.

In [3]:
print(train_df[TARGET].value_counts(normalize=True).round(3))
print(train_df.groupby('loan_grade')[TARGET].mean().round(3))
print(train_df.groupby('cb_person_default_on_file')[TARGET].mean().round(3))

loan_status
0    0.858
1    0.142
Name: proportion, dtype: float64
loan_grade
A    0.049
B    0.102
C    0.135
D    0.594
E    0.625
F    0.611
G    0.818
Name: loan_status, dtype: float64
cb_person_default_on_file
N    0.115
Y    0.299
Name: loan_status, dtype: float64


## Train / validation

In [4]:
X, y = train_df[RAW_FEATURES], train_df[TARGET]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = build_pipeline().fit(X_train, y_train)
val_proba = pipeline.predict_proba(X_val)[:, 1]
threshold = best_threshold(y_val, val_proba)

print(f"Threshold: {threshold}")
for name, value in evaluate(y_val, val_proba, threshold).items():
    print(f"{name:<10} {value:.4f}")

Threshold: 0.34
accuracy   0.9135
precision  0.6966
recall     0.6958
f1         0.6962
roc_auc    0.9102


In [5]:
y_val_pred = (val_proba >= threshold).astype(int)
print(confusion_matrix(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred, target_names=['Low risk (approve)', 'High risk (reject)']))

[[9553  506]
 [ 508 1162]]
                    precision    recall  f1-score   support

Low risk (approve)       0.95      0.95      0.95     10059
High risk (reject)       0.70      0.70      0.70      1670

          accuracy                           0.91     11729
         macro avg       0.82      0.82      0.82     11729
      weighted avg       0.91      0.91      0.91     11729



## Coefficients

Positive coefficient → higher default risk (features are standardized / one-hot).

In [6]:
coefficients = pd.DataFrame({
    'Feature': pipeline.named_steps['preprocess'].get_feature_names_out(),
    'Coefficient': pipeline.named_steps['clf'].coef_[0],
}).sort_values('Coefficient', ascending=False)
coefficients

,Feature,Coefficient
23,cat__loan_grade_D,1.144426
10,cat__person_home_ownership_RENT,1.108969
5,num__loan_percent_income,1.052855
9,num__loan_to_income,0.925637
24,cat__loan_grade_E,0.894828
25,cat__loan_grade_F,0.702931
13,cat__person_home_ownership_OTHER,0.462392
26,cat__loan_grade_G,0.447708
4,num__loan_int_rate,0.367518
18,cat__loan_intent_HOMEIMPROVEMENT,0.343576


## Submission

The competition is scored by ROC-AUC, so we submit probabilities, not 0/1 labels. The final model is refit on all labeled data.

In [7]:
final_pipeline = build_pipeline().fit(X, y)
submission = pd.DataFrame({
    'id': test_df['id'],
    TARGET: final_pipeline.predict_proba(test_df[RAW_FEATURES])[:, 1],
})
submission.to_csv('logregression_submission.csv', index=False)
submission.head()

,id,loan_status
0,58645,0.961506
1,58646,0.007434
2,58647,0.567162
3,58648,0.068063
4,58649,0.308145
